# EDA dirigido + ventas en el tiempo — Superstore

Dos enfoques de EDA:
- **Explorar (abierto)**: abrir el dataset y ver qué aparece
- **Responder (dirigido)**: formular preguntas antes, validarlas con datos

Este notebook sigue el segundo: 5 preguntas formuladas antes de escribir código.


## Setup

In [1]:
import pandas as pd
%matplotlib inline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()
df   = pd.read_csv(ROOT / 'data' / 'external' / 'train.csv')

# Convertir fecha al tipo correcto desde el primer momento
# format='%d/%m/%Y' indica que el dia va primero (08/11/2017 = 8 de noviembre)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')

print(f'Shape: {df.shape}')
print(df.dtypes[['Order Date', 'Sales', 'Category', 'Customer Name']])

Shape: (9800, 18)
Order Date       datetime64[us]
Sales                   float64
Category                    str
Customer Name               str
dtype: object


---
## Las 5 preguntas

1. **¿Qué categoría genera más revenue total?**  
   Hipótesis: Technology lidera por precio unitario alto.

2. **¿Quiénes son los 10 clientes que más gastan?**  
   Hipótesis: pocos clientes concentran una parte desproporcionada del revenue.

3. **¿En qué meses se concentran las ventas?**  
   Hipótesis: noviembre y diciembre son pico por temporada navideña.

4. **¿Qué región tiene el ticket medio más alto?**  
   Hipótesis: West tiene ticket más alto por mercado tecnológico en California.

5. **¿Qué subcategoría tiene el margen de precio más alto (max - min)?**  
   Hipótesis: Machines o Copiers tienen mayor dispersión.


---
## 1 — ¿Qué categoría genera más revenue total?


In [2]:
p1 = (
    df.groupby('Category')['Sales']
    .agg(
        revenue_total  = 'sum',
        num_pedidos    = 'count',
        ticket_medio   = 'mean'
    )
    .sort_values('revenue_total', ascending=False)
    .reset_index()
)
# Redondeo "round" visual — sum() y mean() generan decimales largos innecesarios en el print
p1['revenue_total'] = p1['revenue_total'].round(2)
p1['ticket_medio']  = p1['ticket_medio'].round(2)
print(p1)


ganador = p1.iloc[0]['Category']
print(f'\nRespuesta: {ganador} lidera en revenue total')

          Category  revenue_total  num_pedidos  ticket_medio
0       Technology      827455.87         1813        456.40
1        Furniture      728658.58         2078        350.65
2  Office Supplies      705422.33         5909        119.38

Respuesta: Technology lidera en revenue total


---
## 2 — ¿Quiénes son los 10 clientes que más gastan?


In [3]:
p2 = (
    df.groupby('Customer Name')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'Sales': 'revenue_total'})
)

# Calcular qué porcentaje del revenue total representan los top 10
revenue_global  = df['Sales'].sum()
revenue_top10   = p2['revenue_total'].sum()
pct_top10       = round(revenue_top10 / revenue_global * 100, 2)

p2['revenue_total'] = p2['revenue_total'].round(2)
print(p2.to_string(index=False))
print(f'\nLos top 10 clientes representan el {pct_top10}% del revenue total')

     Customer Name  revenue_total
       Sean Miller       25043.05
      Tamara Chand       19052.22
      Raymond Buch       15117.34
      Tom Ashbrook       14595.62
     Adrian Barton       14473.57
      Ken Lonsdale       14175.23
      Sanjit Chand       14142.33
      Hunter Lopez       12873.30
      Sanjit Engle       12209.44
Christopher Conant       12129.07

Los top 10 clientes representan el 6.8% del revenue total


---
## 3 — ¿En qué meses se concentran las ventas?

`dt.month` agrupa sin importar el año — detecta estacionalidad acumulada.


In [4]:
import plotly.express as px

ventas_mes = (
    df.groupby(df['Order Date'].dt.month)['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'mes', 'Sales': 'revenue_total'})
)

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
ventas_mes['mes_nombre'] = ventas_mes['mes'].apply(lambda x: meses[x-1])

idx_max    = ventas_mes['revenue_total'].idxmax()
mes_pico   = ventas_mes.loc[idx_max, 'mes_nombre']

fig = px.line(
    ventas_mes,
    x='mes_nombre',
    y='revenue_total',
    markers=True,
    title='Revenue mensual acumulado (todos los años)',
    labels={'mes_nombre': 'Mes', 'revenue_total': 'Revenue ($)'}
)

fig.add_annotation(
    x=mes_pico,
    y=ventas_mes.loc[idx_max, 'revenue_total'],
    text=f'Pico: {mes_pico}',
    showarrow=True,
    arrowhead=2,
    yshift=15,
    font=dict(color='steelblue')
)

fig.show()
print(f'Mes con mayor revenue: {mes_pico}')


Mes con mayor revenue: Nov


---
## 4 — ¿Qué región tiene el ticket medio más alto?


In [5]:
p4 = (
    df.groupby('Region')['Sales']
    .agg(
        ticket_medio   = 'mean',
        revenue_total  = 'sum',
        num_pedidos    = 'count'
    )
    .sort_values('ticket_medio', ascending=False)
    .reset_index()
    .round(2)
)

print(p4)
print(f"\nRegión con ticket medio más alto: {p4.iloc[0]['Region']} (${p4.iloc[0]['ticket_medio']:,.2f})")

    Region  ticket_medio  revenue_total  num_pedidos
0    South        243.52      389151.46         1598
1     East        240.40      669518.73         2785
2     West        226.18      710219.68         3140
3  Central        216.36      492646.91         2277

Región con ticket medio más alto: South ($243.52)


---
## 5 — ¿Qué subcategoría tiene mayor dispersión de precio?

Rango = max - min dentro de cada subcategoría.


In [6]:
p5 = (
    df.groupby('Sub-Category')['Sales']
    .agg(
        precio_min  = 'min',
        precio_max  = 'max'
    )
    .reset_index()
)

p5['rango'] = (p5['precio_max'] - p5['precio_min']).round(2)
p5 = p5.sort_values('rango', ascending=False)

print(p5.head(5).to_string(index=False))
print(f"\nMayor dispersión: {p5.iloc[0]['Sub-Category']} (rango ${p5.iloc[0]['rango']:,.2f})")

Sub-Category  precio_min  precio_max    rango
    Machines      11.560    22638.48 22626.92
     Copiers     299.990    17499.95 17199.96
     Binders       0.556     9892.74  9892.18
    Supplies       1.744     8187.65  8185.91
      Phones       2.970     4548.81  4545.84

Mayor dispersión: Machines (rango $22,626.92)


---
## Análisis temporal profundo

`dt.to_period('M')` conserva año+mes (2017-11, 2018-03) — permite ver tendencia real, no solo estacionalidad.


In [7]:
# to_period('M') conserva año y mes juntos: 2017-11, 2018-03, etc.
serie_temporal = (
    df.groupby(df['Order Date'].dt.to_period('M'))['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'periodo', 'Sales': 'revenue'})
)

# pct_change() calcula la variacion porcentual respecto al periodo anterior
serie_temporal['variacion_pct'] = serie_temporal['revenue'].pct_change().mul(100).round(1)

# Convertir periodo a string para graficar
serie_temporal['periodo_str'] = serie_temporal['periodo'].astype(str)

print(serie_temporal.tail(12).to_string(index=False))

periodo     revenue  variacion_pct periodo_str
2018-01  43476.4740          -54.6     2018-01
2018-02  19920.9974          -54.2     2018-02
2018-03  58863.4128          195.5     2018-03
2018-04  35541.9101          -39.6     2018-04
2018-05  43825.9822           23.3     2018-05
2018-06  48190.7277           10.0     2018-06
2018-07  44825.1040           -7.0     2018-07
2018-08  62837.8480           40.2     2018-08
2018-09  86152.8880           37.1     2018-09
2018-10  77448.1312          -10.1     2018-10
2018-11 117938.1550           52.3     2018-11
2018-12  83030.3888          -29.6     2018-12


In [8]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Grafico 1: revenue mensual
ax1.plot(serie_temporal['periodo_str'], serie_temporal['revenue'],
         color='steelblue', linewidth=1.8)
ax1.fill_between(serie_temporal['periodo_str'], serie_temporal['revenue'],
                 alpha=0.12, color='steelblue')
ax1.set_title('Revenue mensual', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.grid(axis='y', alpha=0.3)

# Grafico 2: variacion porcentual mes a mes
colores = ['tomato' if v < 0 else 'seagreen'
           for v in serie_temporal['variacion_pct'].fillna(0)]
ax2.bar(serie_temporal['periodo_str'], serie_temporal['variacion_pct'],
        color=colores, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Variación % respecto al mes anterior', fontsize=11)
ax2.set_ylabel('%')
ax2.grid(axis='y', alpha=0.3)

# Rotar etiquetas del eje x
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

C:\Users\alefe\AppData\Local\Temp\ipykernel_1952\3320435844.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
